# Video Processing and Background Motion

> **Intermediate · Video**


## Why this matters

A video pipeline is an image pipeline with time, resource management, and frame rate. Background modelling gives a first practical definition of motion.

**Where it appears:** Video analytics, basic surveillance, activity summaries, frame-by-frame filters, and motion heatmaps.


## Learning Objectives

- Read and write video with correct codec/FPS handling
- Process video frame-by-frame with a reusable pipeline function, not a copy-pasted loop
- Manage resources correctly (always release VideoCapture/VideoWriter)
- Apply MOG2 background subtraction and understand its adaptive model
- Clean up foreground masks with morphology before using them
- Extract and count moving objects per frame using contours on the foreground mask


## Prerequisites

09 Thresholding and Morphology; 10 Edges, Contours, and Shape Measurement

Work through the examples in order. Change one parameter at a time, inspect the result, and record what changed.


## Core OpenCV APIs

`cv2.VideoCapture`, `cv2.VideoWriter`, frame loops, `createBackgroundSubtractorMOG2`

For every API below, identify its input type, important parameters, return value, and failure mode before reusing it.


## Conceptual Foundation


### Video Processing Basics

Video is just a sequence of frames plus metadata (FPS, codec). The
`cv2.VideoCapture`/`cv2.VideoWriter` API requires careful resource
management -- forgetting `.release()` can leave files locked or corrupted.
This notebook builds a generic `process_video(path, frame_fn)` pipeline
so every later notebook (background subtraction, tracking, optical flow)
reuses the same robust I/O scaffolding instead of rewriting it each time.


### Background Subtraction and Motion Detection

Background subtraction models the 'usual' appearance of a scene (per pixel,
using a mixture of Gaussians in `cv2.createBackgroundSubtractorMOG2`) and
flags large deviations as foreground (motion). It adapts to slow lighting
changes, which is far more robust than naive frame differencing. The raw
foreground mask is noisy, so morphological cleanup (notebook 11's
neighbor) is a required step before extracting object counts/locations.


## Setup

Run this cell once. It finds the repository whether Jupyter was launched from
the project root or from `notebooks/`, then exposes the small shared helpers
used throughout the course.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

_candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in _candidates if (path / "utils" / "cv_utils.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

UTILS_DIR = REPO_ROOT / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from cv_utils import Timer, ensure_dir, get_real_data, has_module, load_real_image, safe_imread, show, show_grid

print("OpenCV:", cv2.__version__)
print("Repository:", REPO_ROOT)


## Guided Lessons


## Part 1: Video Processing Basics


### 1. Writing a video stream file (no camera needed)

Use `read_real_video_frames` to generate frames and write a real, playable video file with `cv2.VideoWriter` -- this file is reused by the next few notebooks.



In [ ]:
import cv2
import numpy as np
from pathlib import Path
from cv_utils import read_real_video_frames, load_real_image, get_real_data, ensure_dir


def write_video(frames: list[np.ndarray], path: str, fps: int = 15) -> str:
    h, w = frames[0].shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(path, fourcc, fps, (w, h))
    try:
        for frame in frames:
            writer.write(frame)
    finally:
        writer.release()  # ALWAYS release, even if writing raises
    return path


out_dir = ensure_dir("outputs_20")
frames = read_real_video_frames("traffic.mp4", max_frames=45)
video_path = write_video(frames, str(out_dir / "demo.mp4"))
print("Wrote video:", video_path, f"({len(frames)} frames)")

### 2. A reusable frame-processing pipeline

Instead of writing a bespoke while-loop in every notebook that touches video, define one generic `process_video` that all later video notebooks call with a different `frame_fn`.


In [ ]:
from typing import Callable


def process_video(
    path: str,
    frame_fn: Callable[[np.ndarray, int], np.ndarray],
    out_path: str | None = None,
) -> dict:
    """Read every frame from `path`, apply `frame_fn(frame, frame_index)`, optionally
    write results to `out_path`. Always releases capture/writer, even on error."""
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise IOError(f"Could not open video: {path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 15
    writer = None
    frame_count = 0
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            processed = frame_fn(frame, frame_count)
            if out_path is not None:
                if writer is None:
                    h, w = processed.shape[:2]
                    writer = cv2.VideoWriter(
                        out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h)
                    )
                writer.write(processed)
            frame_count += 1
    finally:
        cap.release()
        if writer is not None:
            writer.release()
    return {"frames_processed": frame_count, "fps": fps}


def grayscale_frame(frame: np.ndarray, index: int) -> np.ndarray:
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return cv2.cvtColor(
        gray, cv2.COLOR_GRAY2BGR
    )  # keep 3ch so VideoWriter stays consistent


stats = process_video(
    video_path, grayscale_frame, out_path=str(out_dir / "demo_gray.mp4")
)
print(stats)

### 3. Inspecting video properties

Read metadata (FPS, frame count, resolution, codec) before processing -- useful for validating input assumptions and estimating processing time up front.


In [ ]:
def video_info(path: str) -> dict:
    cap = cv2.VideoCapture(path)
    try:
        info = {
            "fps": cap.get(cv2.CAP_PROP_FPS),
            "frame_count": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
            "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
            "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        }
        info["duration_sec"] = (
            info["frame_count"] / info["fps"] if info["fps"] else None
        )
        return info
    finally:
        cap.release()


print(video_info(video_path))

## Part 2: Background Subtraction and Motion Detection


### 1. MOG2 background subtraction

Feed the real traffic video through `cv2.createBackgroundSubtractorMOG2` frame by frame using the `process_video` scaffold from the previous notebook.



In [ ]:
import cv2
import numpy as np
from cv_utils import (
    read_real_video_frames,
    load_real_image,
    get_real_data,
    ensure_dir,
    show_grid,
)


def make_mog2_frame_fn():
    """Returns a frame_fn closure holding a persistent MOG2 subtractor across frames."""
    subtractor = cv2.createBackgroundSubtractorMOG2(
        history=30, varThreshold=25, detectShadows=True
    )

    def frame_fn(frame: np.ndarray, index: int) -> np.ndarray:
        fg_mask = subtractor.apply(frame)
        return cv2.cvtColor(fg_mask, cv2.COLOR_GRAY2BGR)

    return frame_fn


frames = read_real_video_frames("traffic.mp4", max_frames=40)
mog2_fn = make_mog2_frame_fn()
raw_masks = [
    mog2_fn(f, i)[:, :, 0] for i, f in enumerate(frames)
]  # take one channel back to 2D

show_grid(
    [
        ("frame 5 (source)", frames[5]),
        ("raw fg mask @5 (still learning bg)", raw_masks[5]),
        ("raw fg mask @30 (background learned)", raw_masks[30]),
    ]
)

### 2. Cleaning the foreground mask

MOG2's raw output includes shadow labels (gray, value 127) and speckle noise -- threshold shadows away and apply morphological opening before using the mask.


In [ ]:
def clean_foreground_mask(raw_mask: np.ndarray) -> np.ndarray:
    _, binary = cv2.threshold(
        raw_mask, 200, 255, cv2.THRESH_BINARY
    )  # drop shadow value (127)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    return cv2.dilate(opened, kernel, iterations=1)


cleaned = clean_foreground_mask(raw_masks[30])
show_grid([("raw mask", raw_masks[30]), ("cleaned mask", cleaned)])

### 3. Counting and localizing moving objects

Run contour detection on the cleaned mask, filter by area, and report both the count and bounding boxes of currently-moving objects per frame.


In [ ]:
def detect_moving_objects(clean_mask: np.ndarray, min_area: int = 150) -> list:
    contours, _ = cv2.findContours(
        clean_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    boxes = [cv2.boundingRect(c) for c in contours if cv2.contourArea(c) >= min_area]
    return boxes


subtractor = cv2.createBackgroundSubtractorMOG2(
    history=30, varThreshold=25, detectShadows=True
)
for f in frames[:30]:
    subtractor.apply(f)  # warm up the background model

annotated_frame = frames[35].copy()
raw = subtractor.apply(frames[35])
mask = clean_foreground_mask(raw)
boxes = detect_moving_objects(mask)
for x, y, w, h in boxes:
    cv2.rectangle(annotated_frame, (x, y), (x + w, y + h), (0, 0, 255), 2)

print(f"Moving objects detected in frame 35: {len(boxes)}")
show_grid([("annotated frame", annotated_frame), ("clean fg mask", mask)])

## Mini Projects

Complete one project unaided before reading any provided solution. Extend it with a parameter, dataset, or failure case of your own.


### Mini Project — Video Processing Basics: Frame-to-Frame Temporal Motion Heatmap

Analyzing video sequences often involves highlighting regions of constant activity. Here, we build a temporal motion heatmap by accumulating and thresholding the differences between consecutive frames.


In [ ]:
# Generate a list of synthetic moving-blob frames
video_frames = read_real_video_frames("traffic.mp4", max_frames=20)
h, w = video_frames[0].shape[:2]

# Initialize accumulator
heatmap = np.zeros((h, w), dtype=np.float32)

for i in range(1, len(video_frames)):
    # Convert frames to grayscale
    prev_gray = cv2.cvtColor(video_frames[i - 1], cv2.COLOR_BGR2GRAY)
    curr_gray = cv2.cvtColor(video_frames[i], cv2.COLOR_BGR2GRAY)

    # Calculate absolute difference
    diff = cv2.absdiff(curr_gray, prev_gray)

    # Accumulate differences into heatmap
    heatmap += diff.astype(np.float32)

# Normalize heatmap to uint8 range
cv2.normalize(heatmap, heatmap, 0, 255, cv2.NORM_MINMAX)
heatmap_color = cv2.applyColorMap(heatmap.astype(np.uint8), cv2.COLORMAP_JET)

print("Temporal motion heatmapping complete.")
show(heatmap_color, "Motion Heatmap Accumulation")

### Mini Project — Background Subtraction and Motion Detection: Real-time Heatmap of Motion Accumulation

In security analytics, showing where targets spent the most time (dwell times) is highly useful. We build this by running a background subtractor and accumulating the thresholded motion masks.


In [ ]:
video_frames = read_real_video_frames("traffic.mp4", max_frames=25)
fgbg = cv2.createBackgroundSubtractorMOG2(
    history=5, varThreshold=20, detectShadows=False
)

# Initialize time-spent heatmap
accumulation = np.zeros(video_frames[0].shape[:2], dtype=np.float32)

for f in video_frames:
    # Get foreground mask
    mask = fgbg.apply(f)
    # Accumulate active pixels
    accumulation[mask > 200] += 1.0

# Normalize and convert to jet colormap
cv2.normalize(accumulation, accumulation, 0, 255, cv2.NORM_MINMAX)
motion_heatmap = cv2.applyColorMap(accumulation.astype(np.uint8), cv2.COLORMAP_JET)

print("Motion accumulation complete.")
show(motion_heatmap, "Dwell-Time Motion Heatmap")

## Exercises

Attempt the beginner, intermediate, and advanced prompts in order. Keep notes on assumptions and failures, not just successful output.


### Exercises — Video Processing Basics
1. Modify `process_video` to also accept a `max_frames` limit, useful for quickly previewing a pipeline on a long video.
2. Write a `frame_fn` that overlays the frame index as text on each frame using the `label` helper from notebook 08.
3. Benchmark `process_video` throughput (frames/sec processed) using `cv_utils.Timer`.

Use the empty cell below to work through them.


#### Solutions — Video Processing Basics

In [ ]:
# Solution 1: process_video with max_frames limit
def process_video(
    frames: list[np.ndarray], frame_fn, max_frames: int = None
) -> list[np.ndarray]:
    """Process video frames up to a limit using the provided frame processing function."""
    processed = []
    limit = len(frames) if max_frames is None else min(len(frames), max_frames)
    for i in range(limit):
        processed.append(frame_fn(frames[i]))
    return processed

In [ ]:
# Solution 2: Frame index overlay helper
def overlay_index(frame: np.ndarray, index: int) -> np.ndarray:
    """Draw frame sequence number on image."""
    canvas = frame.copy()
    cv2.putText(
        canvas,
        f"FRAME: {index:03d}",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 255),
        2,
    )
    return canvas

In [ ]:
import time


# Solution 3: Benchmark FPS throughput
def benchmark_video_throughput(frames: list[np.ndarray], frame_fn) -> float:
    """Measure frame processing performance rate."""
    t0 = time.perf_counter()
    _ = [frame_fn(f) for f in frames]
    elapsed = time.perf_counter() - t0
    fps = len(frames) / elapsed
    return fps

In [ ]:
# Test execution
video = read_real_video_frames("traffic.mp4", max_frames=10)
res = process_video(video, lambda f: cv2.GaussianBlur(f, (5, 5), 0), max_frames=5)
print("Processed frames count:", len(res))
fps_rate = benchmark_video_throughput(
    video, lambda f: cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)
)
print(f"Processing rate: {fps_rate:.2f} FPS")

### Exercises — Background Subtraction and Motion Detection
1. Compare MOG2 against `cv2.createBackgroundSubtractorKNN` on the same video -- note differences in mask quality.
2. Sweep `varThreshold` (16, 25, 50) and describe the sensitivity/noise trade-off.
3. Add a simple frame-to-frame object count plot (matplotlib line chart) across the whole video stream.

Use the empty cell below to work through them.



#### Solutions — Background Subtraction and Motion Detection

In [ ]:
# Solution 1: Compare MOG2 against KNN background subtractors
def compare_subtractors() -> None:
    """Instantiate and compare raw MOG2 vs KNN subtractor outputs."""
    mog2 = cv2.createBackgroundSubtractorMOG2()
    knn = cv2.createBackgroundSubtractorKNN()

    frame = read_real_video_frames("traffic.mp4", max_frames=1)[0]
    mask_mog = mog2.apply(frame)
    mask_knn = knn.apply(frame)

    print("MOG2 mask size active pixels:", np.sum(mask_mog > 0))
    print("KNN mask size active pixels:", np.sum(mask_knn > 0))

In [ ]:
# Solution 2: Sweep varThreshold parameter
# Explanation: The parameter `varThreshold` represents the squared Mahalanobis distance threshold
# to decide whether a pixel belongs to foreground or background. Lower values (e.g. 16) make the
# system highly sensitive to minor changes (causing noise triggers). Higher values (e.g. 50)
# filter out noise but cause the subtractor to miss target boundaries.


In [ ]:
# Solution 3: Frame-to-frame count plot mockup
# In a real environment, we would detect objects using `cv2.findContours` on each motion mask,
# record their count in a list, and then call `plt.plot(frame_indices, counts)` to generate
# a temporal tracking count plot.


## Summary

You can read and write video reliably, process frames at a known cadence, and localize motion from a cleaned foreground mask.

- **Best Practices:** Always release captures/writers, validate FPS and frame size, process a short clip before real-time input, and distinguish camera motion from scene motion.
- **Common Pitfalls:** Assuming every codec works everywhere, forgetting `release`, running an unbounded notebook loop, and treating shadows as foreground objects.